In [1]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import warnings
import re
import joblib
import string
import os
import time
import json
from dotenv import load_dotenv
from itertools import product

In [2]:
startTime = time.time()
lastTime = startTime
times = dict()

In [3]:
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [4]:
load_dotenv()

True

In [5]:
DATA_DIR = './data/'
STATIC_DIR = '../../static/data/'
BLIZZARD_CLIENT_ID = os.getenv('CLIENT_ID')
BLIZZARD_CLIENT_SECRET = os.getenv('CLIENT_SECRET')

### Choose One ###
#UPDATE_METHOD = "LOAD"
UPDATE_METHOD = "CREATE"

# Functions and Classes

In [6]:
class Profession:
    __all_data = None
    __name = None
    
    def __init__(self, profession):
        self.__all_data = list()
        self.__name = profession
        
    def add(self, itemID, itemName, reagents, crafterName, tag, difficulty, multicraft, quantity, skill,
            rarity, hasReagentQualities, hasEmbellishmentSlot, hasMissiveSlot, hasSafetyComponent, hasCrestSlot):

            self.__all_data.append([self.__name, crafterName, itemID, itemName, None, reagents, tag, rarity, 
                                    difficulty, skill, quantity, multicraft, hasReagentQualities, 
                                    hasEmbellishmentSlot, hasMissiveSlot, hasSafetyComponent, hasCrestSlot])
            
    def get_table(self):
        columns = ['profession', 'character', 'itemID', 'item', 'icon', 'reagents', 'tag', 'rarity', 
                   'difficulty', 'skill1', 'baseQuantity', 'multicraftPercent', 'hasReagentQualities', 
                   'hasEmbellishmentSlot', 'hasMissiveSlot', 'hasSafetyComponent', 'hasCrestSlot']
        dtypes = ['string', 'string', 'int32', 'string', 'string', 'object', 'string', 'string', float, float, 
                  'string', float, bool, bool, bool, bool, bool, bool]
        df = pd.DataFrame(columns=columns, data=self.__all_data)
        return df.astype(dict(zip(columns, dtypes)))
        return df
        
    def set_table(self, df):
        self.__all_data = df.to_numpy()

In [7]:
def getToken(clientID, clientSecret):
    auth = (BLIZZARD_CLIENT_ID, BLIZZARD_CLIENT_SECRET)
    data = {'grant_type':'client_credentials'}
    url = 'https://oauth.battle.net/token'

    response = requests.post(url=url, data=data, auth=auth)
    accessToken = response.json().get('access_token')
    
    return accessToken

In [8]:
def getReagents(professionName, itemID, itemName):
    with open(f'{DATA_DIR}{professionName}/{professionName}_recipes.json') as file:
        recipes = json.load(file)
        
    for recipe in recipes:
        if recipe.get('itemID')==str(itemID) and recipe.get('itemName')==itemName:
            return {int(k):int(v) for k,v in recipe.get('reagents').items()}
    else:
        fileName = f'{DATA_DIR}{professionName}/{professionName}_recipes.json'
        raise KeyError(f'Item ID "{itemID}" with item name "{itemName}" not found in {fileName}')

In [9]:
def getStats(professionName, characterName, itemID, itemName, 
             stats=['skill', 'multicraft', 'resourcefulness', 'ingenuity']):
    assert type(stats)==list, 'stats must be a list of strings'
    for stat in stats:
        assert stat.lower() in ['skill', 'multicraft', 'resourcefulness', 'ingenuity'], f'{stat} is not valid'
    stats = [stat.lower() for stat in stats]
    
    #character profession knowledge points
    knowledgeFile = f'{DATA_DIR}{professionName}/{professionName}_knowledge.csv'
    knowledge = pd.read_csv(knowledgeFile)
    assert characterName in knowledge.columns, f'{characterName} does not know {professionName} [Err:knowledge]'
    knowledge = knowledge.loc[:, ['node', characterName]]
    
    #skill, resourcefulness, ingenuity, and multicraft stats from each node
    nodesFile = f'{DATA_DIR}{professionName}/{professionName}_nodes.json'
    with open(nodesFile) as file:
        nodes = json.load(file)   
        
    #which nodes affect which recipes
    specializationsFile = f'{DATA_DIR}{professionName}/{professionName}_specializations.json'
    with open(specializationsFile) as file:
        specializations = json.load(file)
        for spec in specializations:
            if spec.get('itemID')==str(itemID) and spec.get('itemName')==itemName:
                if spec.get('nodes') is None:
                    raise ValueError(f"{spec} does not exist in {nodesFile}")
                else:
                    specializations = spec.get('nodes')
                
                break
        else:
            raise KeyError(f'Item ID "{itemID}" with name "{itemName}" not found in {specializationsFile}')
        
    stats = dict(zip(stats, np.zeros(len(stats))))
    for stat in stats.keys():
        statValues = nodes.get(stat)
        for spec in specializations:
            try:
                knowledgePoints = int(knowledge.loc[knowledge['node']==spec, characterName])
            except:
                print(f'"{spec}" from {specializationsFile} not found in {knowledgeFile}')
                
            #node not unlocked yet, so don't count it
            if knowledgePoints == -1:
                continue
                
            #apply stat gained per point in spec node
            try:
                stats[stat] += int(statValues.get(spec).get('scaling'))*knowledgePoints
            except:
                print(statValues)
                print(spec)
                raise
                
            #apply stat gained from achieved breakpoints in spec node
            for breakpoint in np.arange(start=0, stop=knowledgePoints+1, step=5):
                stats[stat] += int(statValues.get(spec).get(str(breakpoint)))
        
    
    return stats

In [10]:
def getProfession(professionName, characterName):
    profession = Profession(professionName)
    professionName = professionName.lower()
    
    validProfessions = ['alchemy', 'blacksmithing', 'cooking', 'enchanting', 'engineering', 'inscription', 
                        'jewelcrafting', 'leatherworking', 'tailoring']
    assert professionName.lower() in validProfessions, f'{professionName} is not a valid profession'
    
    #base values from profession level and equipment
    baseValues = pd.read_csv(DATA_DIR+'base_stats.csv')
    baseValues = baseValues.loc[(baseValues['profession']==professionName)&(baseValues['character']==characterName),
                                baseValues.columns]
    assert len(baseValues)==1, 'Issue with base values table'
    baseValues = baseValues.to_dict(orient='records')[0]
    
    #attributes for each recipe (tag, difficulty, base stats, rarity, embellishments, etc)
    items = pd.read_csv(f'{DATA_DIR}{professionName}/{professionName}_items.csv')
    
    #which recipes each character knows
    learned = pd.read_csv(f'{DATA_DIR}{professionName}/{professionName}_learned.csv')
    assert characterName in learned.columns, f'{characterName} does not know {professionName} [Err:learned]'
    learned = learned.loc[:, ['itemID', 'itemName', characterName]]
    
    #set crafter to None if not learned by character, otherwise set to character name
    table = pd.merge(left=items, right=learned, on=['itemID', 'itemName'], how='outer', suffixes=[None, None])
    table['crafter'] = 'None'
    try:
        #since raw data uses TRUE and FALSE to show learned status, table[characterName] returns a Series of
        #True where learned and False where not learned, so only rows where learned are ultimately returned
        #using the table.loc
        table.loc[table[characterName], 'crafter'] = characterName
    except ValueError:
        # don't allow None/NaN, as that means there was a mistmatch in indexes between the items table
        # and the learned tables above
        missing_from_learned = table.loc[table[characterName].isna(), ['itemID', 'itemName']]
        missing_from_items = table.loc[table['tag'].isna(), ['itemID', 'itemName']]
        error_message = ''
        error_message += 'The following items are in the Items table but missing from the Learned table:\n'
        for index, row in missing_from_learned.iterrows():
            error_message += f"Item ID: {row['itemID']}; Item Name: {row['itemName']}\n"
        error_message += '\n'
        error_message += 'The following items are in the Learned table but missing from the Items table:\n'
        for index, row in missing_from_items.iterrows():
            error_message += f"Item ID: {row['itemID']}; Item Name: {row['itemName']}\n"
            
        raise ValueError(error_message) from None
    except:
        print("some other exception happened")
    
    table.loc[:,['difficulty','multicraft','skill']]=table.loc[:,['difficulty','multicraft','skill']].replace({-1:np.nan})
        
    for index, row in table.iterrows():   
        itemID = row['itemID']
        itemName = row['itemName']
        
        #adjust values to allow for np.nan since you can't do float('np.nan')
        try:
            skill = float(row['skill'])
        except:
            skill = np.nan
        
        try:
            multicraft = float(row['multicraft'])
        except:
            multicraft = np.nan
            
        try:
            difficulty = float(row['difficulty'])
        except:
            difficulty = np.nan
        
        if professionName != 'cooking':
            stats = getStats(professionName, characterName, itemID, itemName)
        else:
            stats = dict(zip(['skill', 'multicraft', 'resourcefulness', 'ingenuity'], [0,0,0,0]))
            
        skill += (baseValues.get('level') + baseValues.get('skill') + stats['skill'])
        multicraft += (baseValues.get('multicraft') + stats['multicraft'])
        multicraft = np.round(multicraft/33, 1) #convert to a percent stat rather than stat value
            
        profession.add(itemID = itemID,
                       itemName = itemName,
                       reagents = getReagents(professionName, itemID, itemName),
                       crafterName = row['crafter'],
                       tag = row['tag'],
                       difficulty = difficulty,
                       multicraft = multicraft,
                       quantity = row['quantity'],
                       skill = skill,
                       rarity = row['rarity'],
                       hasReagentQualities = row['hasReagentQualities'],
                       hasEmbellishmentSlot = row['hasEmbellishmentSlot'],
                       hasMissiveSlot = row['hasMissiveSlot'],
                       hasSafetyComponent = row['hasSafetyComponent'],
                       hasCrestSlot = row['hasCrestSlot']
                      )
        
    return profession

In [11]:
def scrapeIcon(itemID):
    """Gets the item icon by scraping WoWhead"""
    url = f'https://www.wowhead.com/item={itemID}'
    soup = BeautifulSoup(requests.get(url).text) 
    
    #string1 finds strings preceded by:   "{itemID}:{" 
    #and are also followed by:    ,"screenshot"
    #the strings cannot include the symbol:   }
    string1 = re.search(r'(?<="'+f'{itemID}'+r'":{)[^}]+(?=,"screenshot")', str(soup)).group()
    
    #string2 searches string1 for strings preceded by:      "icon":"
    #and are also followed by:      ")
    #that only contain a-z, A-Z, 0-9, _, and -
    string2 = re.search(r'(?<="icon":")[\w-]+(?=")', string1).group()
    site = 'https://wow.zamimg.com/images/wow/icons/large/'+string2+'.jpg'
    
    status_code = requests.get(site).status_code
    
    if status_code==200:
        return {itemID: site}
    else:
        return {itemID: None}

In [12]:
def getIcon(itemID, accessToken):
    """Gets the item icon using the Blizzard API"""
    base_url = 'https://us.api.blizzard.com'
    endpoint = f'data/wow/media/item/{itemID}'
    url = f'{base_url}/{endpoint}?namespace=static-us&locale=en_US'
    headers = {'Authorization': f'Bearer {accessToken}'}
    
    rateLimited = True
    while rateLimited:
        try:
            response: requests.models.Response = requests.get(url=url, headers=headers)
        except:
            output: dict[str, int] = {itemID:-1}
                
        if response.status_code == 404:
            output = {itemID:0}
        elif response.status_code == 200:
            output: dict[str, int|str|bool] = response.json()
            try:
                output = output.get('assets')
                output = [asset for asset in output if asset.get('key')=='icon'][0]
                output = {itemID:output['value']}
            except:
                output = {itemID:-1}
        else:
            output: dict[str, int] = {itemID:response.status_code}

        if response.status_code == 429:
            rateLimited = True
            time.sleep(30)
        else:
            rateLimited = False 
            
    return output

In [13]:
def check_id(old_id):
    "scrape WoWhead to get all applicable items as their Quality 3 item IDs"
    url = f'https://www.wowhead.com/item={old_id}&xml'
    html = requests.get(url).text
    soup = BeautifulSoup(html, features='xml')
    name = soup.find('name').text
    text = soup.find('htmlTooltip').text
    
    #check for quality tier information
    if text.find('quality-tier1') >= 0:
        new_id = old_id+2
        new_id_lower = old_id-2
    elif text.find('quality-tier2') >= 0:
        new_id = old_id+1
        new_id_lower = old_id-1
    else: #either its tier3 or it doesn't have tiers, in which use the old_id
        return {old_id: old_id}
    
    
    #wasn't a tier 3 item, so check the calculated id for if the name matches and is tier 3
    #return the new id if it is the same name and tier 3, else return -1 for manual checking
    try:
        url = f'https://www.wowhead.com/item={new_id}&xml'
        html = requests.get(url).text
        soup = BeautifulSoup(html, features='xml')
        if soup.find('name').text == name and soup.find('htmlTooltip').text.find('quality-tier3') >= 0:
            return {old_id: new_id}
    except:
        pass
    
    try:
        url = f'https://www.wowhead.com/item={new_id_lower}&xml'
        html = requests.get(url).text
        soup = BeautifulSoup(html, features='xml')
        if soup.find('name').text == name and soup.find('htmlTooltip').text.find('quality-tier3') >= 0:
            return {old_id: new_id_lower}
    except:
        return {old_id: -1}
        
    return {old_id: -1}

In [14]:
def outcomeQuality(skill, difficulty, tag):
    if tag.lower()[:4]=='gear':
        arr = np.array([1, 0.2*difficulty, 0.5*difficulty, 0.8*difficulty, difficulty])
    else:
        arr = np.array([1, difficulty])
        
    return (skill >= arr).sum()

In [15]:
def updateReagents(reagents: dict, replacementIDs: dict):
    return {replacementIDs.get(reagent, reagent):count for reagent,count in reagents.items()}

In [16]:
curTime = time.time()
times['functions'] = curTime - lastTime
lastTime = curTime

# Initial DataFrames

In [17]:
items_columns = ['itemID', 'item', 'icon', 'tag', 'rarity']
items_dtypes = ['int32', 'string', 'string', 'string', 'string']
items = pd.DataFrame(columns=items_columns)

professions_columns = ['profession', 'itemID', 'reagents', 'hasReagentQualities', 'hasEmbellishmentSlot',
                       'hasMissiveSlot', 'hasSafetyComponent', 'hasCrestSlot']
professions_dtypes = ['string', 'int32', dict, bool, bool, bool, bool, bool]
professions = pd.DataFrame(columns=professions_columns)

crafting_columns = ['itemID', 'difficulty', 'character', 'skill1', 'base_quantity', 'multicraft_percent']
crafting_dtypes = ['int32', 'int16', 'string', float, 'string', float]
crafting = pd.DataFrame(columns=crafting_columns)

In [18]:
alchemy = getProfession('Alchemy', 'Trillithia')
alchemy2 = getProfession('Alchemy', 'Sillik')
blacksmithing = getProfession('Blacksmithing', 'Zarastannil')
blacksmithing2 = getProfession('Blacksmithing', 'Nystelil')
cooking = getProfession('Cooking', 'Trillithia')
enchanting = getProfession('Enchanting', 'Linidel')
enchanting2 = getProfession('Enchanting', 'Mellasona')
engineering = getProfession('Engineering', 'Trillithia')
inscription = getProfession('Inscription', 'Mellasona')
inscription2 = getProfession('Inscription', 'Lindinil')
jewelcrafting = getProfession('Jewelcrafting', 'Nystelil')
jewelcrafting2 = getProfession('Jewelcrafting', 'Zarastannil')
leatherworking = getProfession('Leatherworking', 'Braevele')
leatherworking2 = getProfession('Leatherworking', 'Lindinil')
tailoring = getProfession('Tailoring', 'Linidel')
tailoring2 = getProfession('Tailoring', 'Braevele')

# DataFrame Merging

In [19]:
#concatenate tables
#sort by itemID and skill (descending) so items are paired with higher skill on top
#keep the first entry for each itemID (i.e., the highest skill entry)
#break same skill tie by sorting by name such that primary crafter is at the top
sortCols = ['itemID', 'skill1', 'character']
sortVals = [True, False]
nameAscending = {'alchemy': False,
                 'blacksmithing': False,
                 'enchanting': True,
                 'inscription': False,
                 'jewelcrafting': True,
                 'leatherworking': True,
                 'tailoring': False}

all_data = pd.DataFrame()
for df1, df2, name in [(alchemy, alchemy2, 'alchemy'), (blacksmithing, blacksmithing2, 'blacksmithing'),
                       (enchanting, enchanting2, 'enchanting'), (inscription, inscription2, 'inscription'),
                       (jewelcrafting, jewelcrafting2, 'jewelcrafting'), 
                       (leatherworking, leatherworking2, 'leatherworking'), (tailoring, tailoring2, 'tailoring')]:
    table1 = df1.get_table()
    table2 = df2.get_table()
    
    #recipes known by at least one
    known = pd.concat((table1[table1['character']!='None'], table2[table2['character']!='None']), ignore_index=True)
    
    #recipes not known by both
    unknown = pd.concat((table1[table1['character']=='None'], table2[table2['character']=='None']), ignore_index=True)
    
    #only recipes with known crafters are included, so sort by highest skill, then chosen character order
    #and only take the first (i.e. the character we want to be displayed)
    known = known.sort_values(by=['itemID', 'skill1', 'character'], ascending=[True, False]+[nameAscending[name]])
    known = known.groupby('itemID', as_index=False).first()
    
    #all recipes have crafter as None, so take only the recipe with highest skill if there are duplicates
    unknown = unknown.sort_values(by=['itemID', 'skill1'], ascending=[True, False])
    unknown = unknown.groupby('itemID', as_index=False).first()
    
    #*should* (I think) keep known recipes then unknown recipes in that order. So grouping by itemID should put
    #known ones before unknown and hence only take the prechosen known recipe, unless it was known by no character
    #in which it will only appear once anyway with character=='None', which .first() will collect
    df = pd.concat((known, unknown), ignore_index=True)    
    
    df = df.groupby('itemID', as_index=False).first()
    all_data = pd.concat((all_data, df), ignore_index=True)

In [20]:
#professions with only 1 crafter
all_data = pd.concat((all_data, cooking.get_table()), ignore_index=True)
all_data = pd.concat((all_data, engineering.get_table()), ignore_index=True)
all_data = all_data.reset_index(drop=True)

# Manual Adjustments

In [21]:
fixes = {'"Magically ""Infinite"" Messenger"': 'Magically "Infinite" Messenger'}
all_data['item'] = all_data['item'].apply(lambda x: fixes.get(x, x))

In [22]:
curTime = time.time()
times['base frames'] = curTime - lastTime
lastTime = curTime

# Items DataFrame

In [23]:
#single dataframe of all items, including those listed in reagents
columns = ['itemID', 'item', 'icon']
items = all_data.loc[:, ['itemID', 'item', 'icon']]
items = items.drop_duplicates()
        
for index, row in tqdm(all_data.iterrows(), total=len(all_data)):
    for reagent in row['reagents'].keys():
        reagent = int(reagent) #change type from str to int so they match correctly
        if reagent not in items.loc[:, 'itemID'].to_numpy():
            url = f'https://www.wowhead.com/item={reagent}?xml'
            html = requests.get(url).text
            soup = BeautifulSoup(html, features='xml')
            name = soup.find('name').text
            df = pd.DataFrame(columns=columns, data=[[reagent, name, None]])
            items = pd.concat((items, df))

100%|██████████| 698/698 [01:03<00:00, 11.02it/s]


In [24]:
curTime = time.time()
times['items lookup'] = curTime - lastTime
lastTime = curTime

# Item Icons

In [25]:
icon_file = STATIC_DIR+'icons.pkl'

if not os.path.isfile(icon_file) or UPDATE_METHOD == "CREATE":
    accessToken = getToken(BLIZZARD_CLIENT_ID, BLIZZARD_CLIENT_SECRET)
    
    num_cores = joblib.cpu_count()
    all_jobs = [joblib.delayed(getIcon)(itemID, accessToken) for itemID in items['itemID'].values]
    print(f'Num Tasks: {len(all_jobs)}')
    results = joblib.Parallel(n_jobs=num_cores, verbose=10)(all_jobs)
    
    icon_links = {int(k):v for d in results for k,v in d.items()}
    
    icons_df = pd.DataFrame()
    icons_df['itemID'] = icon_links.keys()
    icons_df['link'] = icon_links.values()
    icons_df.to_pickle(icon_file)
elif os.path.isfile(icon_file) and UPDATE_METHOD == "LOAD":
    icon_links = pd.read_pickle(icon_file)
    icon_links = dict(zip(icon_links['itemID'].values, icon_links['link']))

Num Tasks: 789


[Parallel(n_jobs=24)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:   11.0s
[Parallel(n_jobs=24)]: Done  13 tasks      | elapsed:   13.6s
[Parallel(n_jobs=24)]: Done  24 tasks      | elapsed:   14.1s
[Parallel(n_jobs=24)]: Done  37 tasks      | elapsed:   15.6s
[Parallel(n_jobs=24)]: Done  50 tasks      | elapsed:   16.1s
[Parallel(n_jobs=24)]: Done  65 tasks      | elapsed:   17.7s
[Parallel(n_jobs=24)]: Done  80 tasks      | elapsed:   18.6s
[Parallel(n_jobs=24)]: Done  97 tasks      | elapsed:   19.9s
[Parallel(n_jobs=24)]: Done 114 tasks      | elapsed:   21.4s
[Parallel(n_jobs=24)]: Done 133 tasks      | elapsed:   22.9s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:   24.3s
[Parallel(n_jobs=24)]: Done 173 tasks      | elapsed:   25.8s
[Parallel(n_jobs=24)]: Done 194 tasks      | elapsed:   27.5s
[Parallel(n_jobs=24)]: Done 217 tasks      | elapsed:   29.4s
[Parallel(n_jobs=24)]: Done 240 tasks      | elapsed:  

In [26]:
#update icons in dataframes
all_data['icon'] = all_data['itemID'].map(icon_links)
items['icon'] = items['itemID'].map(icon_links)

In [27]:
#ensure all items have icons
df = all_data.loc[(all_data['icon'].isna())|(all_data['icon']==-1)|(all_data['icon']==0), ['itemID', 'icon']]
assert(len(df)==0)

In [28]:
curTime = time.time()
times['icons'] = curTime - lastTime
lastTime = curTime

# Add Difficulties

In [30]:
#all mats rank 2
all_data['skill2'] = all_data['skill1']+all_data['difficulty']*0.4

extraDiff = {
    '': 0,
    'missive': 5,
    'embellishment': 5,
    'adventurer': 60,
    'veteran': 100,
    'hero': 10,
    'myth': 20,
    'combatant': 0,
    'aspirant': 50,
    'gladiator': 150
}

stats = ('', 'missive')
effects = ('', 'embellishment')
pve_levels = ('', 'adventurer', 'veteran', 'hero', 'myth')
pvp_levels = ('', 'combatant', 'aspirant', 'gladiator')
all_combos = product(pve_levels+pvp_levels, stats, effects)

for combo in all_combos:
    suffix = '_'.join([c for c in combo if c != '']).strip('_')
    if suffix != '':
        suffix = '_' + suffix
    
    try:
        modifier = sum([extraDiff.get(mod) for mod in combo])
    except:
        raise KeyError(f'Invalid combination: {combo}') from None
        
    all_data['difficulty'+suffix] = all_data['difficulty'] + modifier
    all_data['rank1mats_outcome'+suffix] = all_data.apply(lambda row: outcomeQuality(row['skill1'],
                                                                                     row['difficulty'+suffix],
                                                                                     row['tag']), axis=1)
    all_data['rank2mats_outcome'+suffix] = all_data.apply(lambda row: outcomeQuality(row['skill2'],
                                                                                     row['difficulty'+suffix],
                                                                                     row['tag']), axis=1)

In [31]:
curTime = time.time()
times['difficulties'] = curTime - lastTime
lastTime = curTime

# Proper Case tag field

In [32]:
all_data['tag'] = all_data['tag'].apply(lambda x: x.title() if x != "gear (pvp)" else "Gear (PvP)")

In [33]:
curTime = time.time()
times['text formatting'] = curTime - lastTime
lastTime = curTime

# Remove pd.NA and Sorting

In [34]:
all_data = all_data.reset_index(drop=True)

all_data['character'] = all_data['character'].replace({pd.NA:None, 'None':None})
all_data = all_data.sort_values(by=['profession', 'item'], ascending=[True, True])

assert(len(all_data['character'].unique()==8) or len(all_data['character'].unique()==9))

In [35]:
curTime = time.time()
times['trimming and sorting'] = curTime - lastTime
lastTime = curTime

# File Saving

In [36]:
items.to_pickle(STATIC_DIR+'items_midnight.pkl')

In [37]:
all_data.to_pickle(STATIC_DIR+'data_midnight.pkl')

In [38]:
curTime = time.time()
times['saving'] = curTime - lastTime
lastTime = curTime

# File Loading

In [39]:
items = pd.read_pickle(STATIC_DIR+'items_midnight.pkl')

In [40]:
all_data = pd.read_pickle(STATIC_DIR+'data_midnight.pkl')

# Times

In [ ]:
curTime = time.time()
times['total'] = curTime - startTime

display(times)

# Issue Testing

In [41]:
all_data

,itemID,profession,character,item,icon,reagents,tag,rarity,difficulty,skill1,...,rank2mats_outcome_gladiator,difficulty_gladiator_embellishment,rank1mats_outcome_gladiator_embellishment,rank2mats_outcome_gladiator_embellishment,difficulty_gladiator_missive,rank1mats_outcome_gladiator_missive,rank2mats_outcome_gladiator_missive,difficulty_gladiator_missive_embellishment,rank1mats_outcome_gladiator_missive_embellishment,rank2mats_outcome_gladiator_missive_embellishment
13,241299,Alchemy,Sillik,Amani Extract,https://render.worldofwarcraft.com/us/icons/56...,"{240991: 5, 236761: 8, 236774: 3}",Potion,common,530.0,320.0,...,1,685.0,1,1,685.0,1,1,690.0,1,1
32,245650,Alchemy,Trillithia,Bouquet of Herbs,https://render.worldofwarcraft.com/us/icons/56...,"{242651: 1, 243599: 20, 243602: 4}",Transmutation,uncommon,465.0,190.0,...,1,620.0,1,1,620.0,1,1,625.0,1,1
30,242650,Alchemy,Trillithia,Box of Rocks,https://render.worldofwarcraft.com/us/icons/56...,"{238525: 8, 242651: 1, 238520: 18, 238518: 18}",Transmutation,uncommon,465.0,190.0,...,1,620.0,1,1,620.0,1,1,625.0,1,1
22,241319,Alchemy,Trillithia,Cauldron of Sin'dorei Flasks,https://render.worldofwarcraft.com/us/icons/56...,"{236780: 1, 242651: 5, 251285: 4, 240991: 20, ...",Cauldron,epic,605.0,365.0,...,1,760.0,1,1,760.0,1,1,765.0,1,1
4,241281,Alchemy,Trillithia,Composite Flora,https://render.worldofwarcraft.com/us/icons/56...,"{236951: 4, 236950: 4, 236761: 6, 236776: 4}",Transmutation,uncommon,365.0,190.0,...,1,520.0,1,1,520.0,1,1,525.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527,239685,Tailoring,None,Thalassian Competitor's Cloth Tunic,https://render.worldofwarcraft.com/us/icons/56...,"{256559: 5, 251665: 2, 236952: 4, 238523: 4, 2...",Gear (PvP),uncommon,280.0,250.0,...,4,435.0,3,4,435.0,3,4,440.0,3,4
548,267056,Tailoring,Linidel,Thalassian Enchanter's Bonnet,https://render.worldofwarcraft.com/us/icons/56...,"{251691: 2, 236951: 5, 245345: 20, 239198: 8, ...",Gear (Profession),epic,415.0,215.0,...,3,570.0,2,3,570.0,2,3,575.0,2,3
549,267060,Tailoring,Linidel,Thalassian Herbalist's Cowl,https://render.worldofwarcraft.com/us/icons/56...,"{251691: 2, 236951: 5, 245345: 20, 239198: 8, ...",Gear (Profession),epic,415.0,215.0,...,3,570.0,2,3,570.0,2,3,575.0,2,3
550,267062,Tailoring,Linidel,Thalassian Tailor's Threads,https://render.worldofwarcraft.com/us/icons/56...,"{251691: 2, 236952: 5, 245345: 20, 239201: 8, ...",Gear (Profession),epic,415.0,215.0,...,3,570.0,2,3,570.0,2,3,575.0,2,3
